# DICOM → BIDS with HeuDiConv

This notebook converts raw MRI DICOMs into a [BIDS](https://bids.neuroimaging.io/) dataset using [HeuDiConv](https://heudiconv.readthedocs.io/).

**The workflow (two passes):**

1. **Discovery pass** – run HeuDiConv with the `convertall` heuristic and `-c none`. Nothing is converted; it just writes `dicominfo.tsv`, a table of every series in your scan.
2. **Look at the table, write a heuristic** – a small Python file that says "series named *mprage* → `anat/..._T1w`", etc.
3. **Conversion pass** – run HeuDiConv again with your heuristic to produce NIfTI + JSON sidecars + BIDS files.
4. **QC + deface** – inspect the output, deface the T1w/T2w with pydeface (keeping an unfaced backup outside the dataset), and check the before/after images.
5. **Copy** the defaced `sub-*` folders into your main dataset and run the BIDS validator there.

> 🖥️ **Where to run this:** run the notebook *on the server*, since the DICOMs and the Singularity image live there.
>
> 📦 **Get the singularity images once (pin a version so your conversions are reproducible):**
> 
> `singularity pull heudiconv_1.3.2.sif docker://nipy/heudiconv:1.3.2`
> 
> `singularity pull pydeface.sif docker://poldracklab/pydeface:latest`
> 
> `singularity pull bids-validator.sif docker://bids/validator`

> ⚠️ **PHI warning:** DICOM headers contain identifying info. HeuDiConv keeps copies of headers in `<output>/.heudiconv/`. Don't push that folder to GitHub / share it without checking.

## 0. Configure

Edit the cell below, then run everything top to bottom. Only this cell and the heuristic rules in Step 3 should need changes.

**`DICOM_TEMPLATE`** tells HeuDiConv how your DICOM folders are laid out, relative to `DICOM_ROOT`. Use `{subject}` (and `{session}` if you have sessions) as placeholders. Examples:

| Your folder layout | `DICOM_TEMPLATE` |
|---|---|
| `raw/sub01/SeriesA/img001.dcm` | `{subject}/*/*` |
| `raw/sub01/ses1/SeriesA/img001.dcm` | `{subject}/{session}/*/*` |
| `raw/sub01/*.dcm` (flat) | `{subject}/*` |

In [ ]:
from pathlib import Path

# ---- EDIT THESE LINES -------------------------------------------------------

# ---- Paths -----------------------------------------------------------------
DICOM_ROOT = Path("/jukebox/norman/emcdevitt/pygers_workshop/pygers-meg/sample-study/data/auditory-oddball")      # folder containing one folder per subject
BIDS_OUT   = Path("/jukebox/norman/emcdevitt/pygers_workshop/pygers-meg/sample-study/data/bids_staging")    # STAGING folder for the conversion (created if missing)
# Path to your larger BIDS dataset (the one with /meg). Only used by the optional copy step; leave None to skip.
MAIN_DATASET = Path("/jukebox/norman/emcdevitt/pygers_workshop/pygers-meg/sample-study/data/bids")

# ---- Paths to singularity images -----------------------------------------------------------------
SIF_PATH     = Path("/jukebox/norman/emcdevitt/toolbox/heudiconv_1.3.2.sif")
PYDEFACE_SIF = Path("/jukebox/norman/emcdevitt/toolbox/pydeface.sif")
VALIDATOR_SIF = Path("/jukebox/norman/emcdevitt/toolbox/bids-validator.sif")

# ---- Layout of your DICOMs (relative to DICOM_ROOT) -------------------------
DICOM_TEMPLATE = "sub-{subject}/raw/mri/dcm/*.dcm"

# ---- Which subject(s) / session to convert ----------------------------------
# Subject label must match the folder name in DICOM_ROOT; BIDS output labels
# can only contain letters and numbers (no underscores/dashes).
SUBJECTS = ["001"]
SESSION  = "01"        # -> ses-01. Use None if there are no sessions


In [ ]:
# ---- How to run HeuDiConv ---------------------------------------------------
RUNTIME = "singularity"          # "singularity" (also covers Apptainer), "docker", or "local"
DOCKER_IMAGE = "nipy/heudiconv:latest"      # only used if RUNTIME == "docker"
EXTRA_BINDS = []                 # extra folders to mount, e.g. ["/scratch", "/data/shared"]
OVERWRITE = True                # True -> redo conversions that already exist

# Add _run-01, _run-02... to every filename? Leave False if each scan is acquired once
# per session (cleaner names). Set True if a scan is ever repeated, so repeats get numbered.
ADD_RUN = False

# These scans will join a larger BIDS dataset, so do NOT create top-level files
# (dataset_description.json, participants.tsv/json, README, CHANGES). True -> heudiconv "-b notop".
SKIP_TOP_LEVEL = True

HEURISTIC_PATH = BIDS_OUT.parent / "code" / "heuristic.py"

## 1. Check your environment

In [ ]:
import os, datetime, shutil, subprocess, sys

def sh(cmd, check=False):
    """Run a command, stream output live, return exit code."""
    print("$", " ".join(map(str, cmd)), "\n")
    proc = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}")
    return rc

SING = shutil.which("apptainer") or shutil.which("singularity")   # either name works the same

if RUNTIME == "singularity":
    print("singularity/apptainer:", SING or "NOT FOUND  <-- on many clusters: `module load singularity` BEFORE starting Jupyter")
    print("image exists:", SIF_PATH.exists(), "->", SIF_PATH)
    if SING and SIF_PATH.exists():
        sh([SING, "run", "--cleanenv", SIF_PATH, "--version"])
elif RUNTIME == "docker":
    print("docker:", shutil.which("docker") or "NOT FOUND")
else:
    print("heudiconv:", shutil.which("heudiconv") or "NOT FOUND  <-- pip install heudiconv")
    print("dcm2niix: ", shutil.which("dcm2niix")  or "NOT FOUND  <-- pip install dcm2niix")

print("\nDICOM_ROOT exists:", DICOM_ROOT.exists())

## 2. Peek at your DICOMs

A quick sanity check that the paths/template are right *before* running anything heavy.

In [ ]:
import glob

for sub in SUBJECTS:
    pattern = DICOM_TEMPLATE.format(subject=sub, session=SESSION or "*")
    files = sorted(glob.glob(str(DICOM_ROOT / pattern)))
    print(f"sub-{sub}: {len(files)} files match  {pattern}")
    if not files:
        print("   -> nothing matched. Check DICOM_ROOT, SUBJECTS and DICOM_TEMPLATE.")
        continue
    # Try to read one header (pydicom is a dependency of heudiconv's stack, but optional here)
    try:
        import pydicom
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        for tag in ["PatientID", "StudyDate", "Manufacturer", "MagneticFieldStrength", "SeriesDescription"]:
            print(f"   {tag}: {ds.get(tag, 'n/a')}")
    except Exception as e:
        print("   (couldn't read a header with pydicom:", e, ")")

## 3. Discovery pass → `dicominfo.tsv`

This uses the built-in `convertall` heuristic with `-c none` so **nothing is converted**; HeuDiConv only catalogs the series.

In [ ]:
def heudiconv_cmd(subject, heuristic, converter, bids=True):
    """Build the heudiconv command for the chosen runtime."""
    BIDS_OUT.mkdir(parents=True, exist_ok=True)
    args = ["-s", subject, "-f", heuristic, "-c", converter]
    if bids and converter != "none":
        args += ["-b", "notop"] if SKIP_TOP_LEVEL else ["-b"]
    if SESSION:
        args += ["-ss", SESSION]
    if OVERWRITE:
        args += ["--overwrite"]

    if RUNTIME == "local":
        return ["heudiconv", "-d", str(DICOM_ROOT / DICOM_TEMPLATE), "-o", str(BIDS_OUT)] + args

    # Containers: mount dicoms read-only at /data, output at /base, heuristic folder at /code
    mounts = [(DICOM_ROOT.resolve(), "/data", "ro"), (BIDS_OUT.resolve(), "/base", "rw")]
    if heuristic != "convertall":
        HEURISTIC_PATH.parent.mkdir(parents=True, exist_ok=True)
        mounts.append((HEURISTIC_PATH.parent.resolve(), "/code", "ro"))
        args[args.index("-f") + 1] = f"/code/{HEURISTIC_PATH.name}"
    for extra in EXTRA_BINDS:
        mounts.append((Path(extra).resolve(), str(extra), "rw"))

    tool_args = ["-d", f"/data/{DICOM_TEMPLATE}", "-o", "/base"] + args

    if RUNTIME == "singularity":
        binds = [x for src, dst, mode in mounts for x in ("-B", f"{src}:{dst}:{mode}")]
        # --cleanenv keeps your host environment variables from leaking into the container
        return [SING, "run", "--cleanenv"] + binds + [str(SIF_PATH)] + tool_args

    binds = [x for src, dst, mode in mounts for x in ("-v", f"{src}:{dst}:{mode}")]
    return ["docker", "run", "--rm", "-u", f"{os.getuid()}:{os.getgid()}"] + binds + [DOCKER_IMAGE] + tool_args

for sub in SUBJECTS:
    sh(heudiconv_cmd(sub, "convertall", "none"))

In [ ]:
import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 50, "display.max_rows", 200)

tsvs = sorted(BIDS_OUT.glob(".heudiconv/**/dicominfo*.tsv"))
print("Found:", *[str(t.relative_to(BIDS_OUT)) for t in tsvs], sep="\n  ")

if tsvs:
    info = pd.read_csv(tsvs[0], sep="\t")
    cols = ["series_id", "protocol_name", "series_description", "dim1", "dim2", "dim3", "dim4",
            "TR", "TE", "is_derived", "image_type"]
    display(info[[c for c in cols if c in info.columns]])

### 👀 Read the table above

For each series you want in BIDS, note what makes it identifiable:

* `protocol_name` / `series_description` – usually the best handle (e.g. `T1_MPRAGE`, `rest_bold`)
* `dim4 > 1` – 4D data (BOLD, DWI)
* `is_derived` / `image_type` – filter out scanner-generated derived images (e.g. MPR recons, ADC maps)
* Localizers/scouts – skip them

Then adjust the rules in the next cell.

## 4. Write the heuristic

The heuristic is a Python file with an `infotodict(seqinfo)` function. It receives one record per series and returns a dict mapping **BIDS filename templates → list of series IDs**.

The rules below cover common cases (T1w, T2w, FLAIR, resting/task BOLD, DWI, spin-echo fieldmaps). **Edit the keyword matches to fit the `protocol_name`s in your table.**

In [ ]:
HEURISTIC = r"""
# Auto-generated by the notebook. Edit the matching rules to fit your scanner protocols.

SES_DIR = "__SESDIR__"   # "{session}/" or ""
SES_TAG = "__SESTAG__"   # "_{session}" or ""
RUN = "__RUN__"          # "_run-{item:02d}" or ""

import re
ACQ_RE = re.compile(r"acq-([0-9a-z.]+)")   # picks up e.g. acq-0.8mm from the protocol name

# Task name -> keyword(s) found in protocol_name (lowercase)
TASKS = {
    "rest": ["rest", "rsfmri", "resting"],
    # "nback": ["nback", "n-back"],
}

# Fieldmaps only: to auto-fill \"IntendedFor\" in fieldmap sidecars, uncomment this block.
# POPULATE_INTENDED_FOR_OPTS = {
#     "matching_parameters": "ModalityAcquisitionLabel",
#     "criterion": "Closest",
# }

def create_key(template, outtype=("nii.gz",), annotation_classes=None):
    if not template:
        raise ValueError("Template must be a valid format string")
    return template, outtype, annotation_classes

def acq_tag(name):
    # BIDS labels must be alphanumeric, so "0.8mm" -> "0p8mm"
    m = ACQ_RE.search(name)
    if not m:
        return ""
    return "_acq-" + re.sub(r"[^0-9a-z]", "", m.group(1).replace(".", "p"))

def anat(suffix, name=""):
    return create_key("sub-{subject}/" + SES_DIR + "anat/sub-{subject}" + SES_TAG + acq_tag(name) + RUN + "_" + suffix)

def func(task, suffix="bold"):
    return create_key("sub-{subject}/" + SES_DIR + "func/sub-{subject}" + SES_TAG +
                      "_task-" + task + RUN + "_" + suffix)

def dwi_key():
    return create_key("sub-{subject}/" + SES_DIR + "dwi/sub-{subject}" + SES_TAG + RUN + "_dwi")

def fmap_epi(direction):
    return create_key("sub-{subject}/" + SES_DIR + "fmap/sub-{subject}" + SES_TAG +
                      "_dir-" + direction + RUN + "_epi")

def has(text, words):
    return any(w in text for w in words)

def infotodict(seqinfo):
    info = {}
    def add(key, s):
        info.setdefault(key, []).append(s.series_id)

    for s in seqinfo:
        p = (s.protocol_name or "").lower()
        d = (s.series_description or "").lower()
        name = p + " " + d
        itype = [str(x).upper() for x in (s.image_type or ())]

        # --- skip junk ---------------------------------------------------
        if s.is_derived or "DERIVED" in itype:
            continue
        if has(name, ["localizer", "scout", "survey", "aahead", "report", "phoenix"]):
            continue

        # --- fieldmaps (spin-echo EPI, AP/PA) -------------------------------
        if has(name, ["se_epi", "topup", "se-epi", "fmap"]) and s.dim4 <= 2:
            if has(name, ["_pa", "pa_", "-pa", "_p-a", "p>>a"]):
                add(fmap_epi("PA"), s); continue
            if has(name, ["_ap", "ap_", "-ap", "_a-p", "a>>p"]):
                add(fmap_epi("AP"), s); continue

        # --- DWI --------------------------------------------------------------
        if has(name, ["dwi", "dti", "diffusion"]) and s.dim4 > 1:
            add(dwi_key(), s); continue

        # --- functional -------------------------------------------------------
        if s.dim4 > 1:
            matched = False
            for task, kws in TASKS.items():
                if has(name, kws):
                    add(func(task), s); matched = True; break
            if matched:
                continue

        # --- anatomical -------------------------------------------------------
        if s.dim4 == 1:
            if has(name, ["flair"]):
                add(anat("FLAIR", name), s); continue
            if has(name, ["t2w", "t2_space", "t2_tse", "t2 space", "t2_"]):
                add(anat("T2w", name), s); continue
            if has(name, ["mprage", "t1w", "t1_", "spgr", "bravo", "t1 "]):
                add(anat("T1w", name), s); continue

        print("UNMATCHED series (not converted):", s.series_id, "|", s.protocol_name)

    return info
"""

if SESSION:
    HEURISTIC = HEURISTIC.replace("__SESDIR__", "{session}/").replace("__SESTAG__", "_{session}")
else:
    HEURISTIC = HEURISTIC.replace("__SESDIR__", "").replace("__SESTAG__", "")
HEURISTIC = HEURISTIC.replace("__RUN__", "_run-{item:02d}" if ADD_RUN else "")

HEURISTIC_PATH.parent.mkdir(parents=True, exist_ok=True)
HEURISTIC_PATH.write_text(HEURISTIC)
print("Wrote", HEURISTIC_PATH)

### Dry-run the heuristic against `dicominfo.tsv`

Before converting, check what the heuristic *would* do with the series from your discovery pass. This runs entirely in Python, so it's instant and safe.

In [ ]:
import ast, importlib.util
from collections import namedtuple

if tsvs:
    spec = importlib.util.spec_from_file_location("heuristic", HEURISTIC_PATH)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    df = pd.read_csv(tsvs[0], sep="\t").fillna("")
    Row = namedtuple("Row", df.columns.tolist())
    seqinfo = []
    for r in df.to_dict("records"):
        it = r.get("image_type", "")
        if isinstance(it, str) and it.startswith("("):
            it = ast.literal_eval(it)
        r["image_type"] = it
        r["is_derived"] = str(r.get("is_derived", "")).lower() == "true"
        seqinfo.append(Row(**r))

    result = mod.infotodict(seqinfo)
    print("\nPlanned conversions:\n")
    for (tmpl, _, _), ids in result.items():
        print(f"  {tmpl}\n      <- series {ids}")
else:
    print("Run the discovery pass first.")

## 5. Convert

Runs HeuDiConv with your heuristic and `dcm2niix`, producing NIfTI files and JSON sidecars in BIDS layout.

With `SKIP_TOP_LEVEL = True` the command uses `-b notop`, so HeuDiConv **will not** write `dataset_description.json`, `participants.tsv`, `participants.json`, `README`, or `CHANGES`; your main dataset keeps its own. It still writes per-session `sub-*_ses-*_scans.tsv` files, which are normal BIDS and merge fine alongside the MEG data.

> The hidden `.heudiconv/` folder (which holds copies of DICOM headers) is always created in the output folder. That is why `BIDS_OUT` is a **staging** folder: convert there, then copy only the `sub-*` folders into the main dataset (Step 8) and leave `.heudiconv/` behind.

In [ ]:
for sub in SUBJECTS:
    sh(heudiconv_cmd(sub, str(HEURISTIC_PATH), "dcm2niix"))

## 6. Inspect the result

In [ ]:
def tree(root, max_depth=4):
    root = Path(root)
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        if any(part.startswith(".") for part in rel.parts):   # hide .heudiconv, .git, etc.
            continue
        if len(rel.parts) > max_depth:
            continue
        print("  " * (len(rel.parts) - 1) + ("📁 " if p.is_dir() else "📄 ") + p.name)

tree(BIDS_OUT)

In [ ]:
import json

niftis = sorted(BIDS_OUT.glob("sub-*/**/*.nii.gz"))
print(f"{len(niftis)} NIfTI files\n")

# Sidecar sanity checks
for nii in niftis:
    side = nii.with_name(nii.name.replace(".nii.gz", ".json"))
    if not side.exists():
        print("MISSING sidecar:", side.name); continue
    meta = json.loads(side.read_text())
    problems = []
    if nii.name.endswith("_bold.nii.gz"):
        if "TaskName" not in meta:           problems.append("no TaskName")
        if "RepetitionTime" not in meta:     problems.append("no RepetitionTime")
    if "fmap" in nii.parts and "IntendedFor" not in meta:
        problems.append("no IntendedFor (fieldmap not linked to a scan)")
    print(("⚠️ " if problems else "✅ ") + nii.name, "|", ", ".join(problems))

## 7. Deface the anatomical images

T1w/T2w scans contain the participant's face, so they should be defaced before they go into a shared dataset. This step uses [**pydeface**](https://github.com/poldracklab/pydeface), which registers each image to a template with FSL's FLIRT and zeroes out the face region. Pydeface needs **FSL**, so the easiest route on a server is its container:

```
singularity pull pydeface.sif docker://poldracklab/pydeface:latest   # then pin a specific tag from Docker Hub
```

How it works here:
* **Backup first.** The identifiable original of each image is copied to `BACKUP_DIR` before anything is changed, and the defaced image then replaces it **in place** in the staging folder (so the BIDS filenames stay the same). Keep `BACKUP_DIR` **outside** any dataset you'll share, and secure it like the raw DICOMs.
* **Each image is defaced independently.** Your T1w and T2w have different fields of view, so reusing one mask for both isn't safe.
* **A log** (`.deface_log.tsv`, hidden in the staging folder) records what's been done, so re-running the cell won't deface things twice.
* **Always look at the QC images below.** Defacing can occasionally clip brain, especially on T2w. If one fails, try `--cost corratio` or `--nocleanup` flags (see `pydeface --help`), or a different tool such as `mri_reface` or `mri_deface`.

In [ ]:
# ---- Defacing config ----------------------------------------------------------
DEFACE_SUFFIXES  = ["T1w", "T2w"]            # add "FLAIR" etc. if needed
PYDEFACE_RUNTIME = "singularity"             # "singularity" (container with FSL) or "local" (pydeface + FSL installed)

# Only if auto-detection fails: full path to the pydeface executable INSIDE the image
# (the diagnostics printed in Step 7 will show where it is), e.g. "/usr/local/miniconda/bin/pydeface"
PYDEFACE_EXECUTABLE = None
BACKUP_DIR       = BIDS_OUT.parent / "unfaced_backup"   # KEEP OUTSIDE ANY SHARED DATASET
REDEFACE         = False                     # True -> deface again even if the log says it's done

# ---- Environment check -----------------------------------------------------------
if PYDEFACE_RUNTIME == "singularity":
    print("singularity/apptainer:", SING or "NOT FOUND (module load singularity before starting Jupyter)")
    print("pydeface image exists:", PYDEFACE_SIF.exists(), "->", PYDEFACE_SIF)
else:
    print("pydeface:", shutil.which("pydeface") or "NOT FOUND  <-- pip install pydeface")
    print("FSLDIR:  ", os.environ.get("FSLDIR") or "NOT SET  <-- pydeface needs FSL")
    print("flirt:   ", shutil.which("flirt") or "NOT FOUND")

In [ ]:
LOG = BIDS_OUT / ".deface_log.tsv"
done = {l.split("\t")[0] for l in LOG.read_text().splitlines()[1:]} if LOG.exists() else set()

def _container_cmd(mode, bind_dir, tail):
    """Build the apptainer command for one of three ways of launching pydeface."""
    b = ["-B", f"{Path(bind_dir).resolve()}:/work"]
    if mode == "exec_name":          # pydeface is on the image's PATH
        return [SING, "exec", "--cleanenv"] + b + [str(PYDEFACE_SIF), "pydeface"] + tail
    if mode == "exec_path":          # explicit path inside the image
        return [SING, "exec", "--cleanenv"] + b + [str(PYDEFACE_SIF), PYDEFACE_EXECUTABLE] + tail
    if mode == "run":                # the image's entrypoint IS pydeface
        return [SING, "run", "--cleanenv"] + b + [str(PYDEFACE_SIF)] + tail
    raise ValueError(mode)

def _probe(mode):
    try:
        r = subprocess.run(_container_cmd(mode, ".", ["--help"]), capture_output=True, text=True, timeout=180)
    except Exception:
        return False
    return r.returncode == 0 and "usage" in (r.stdout + r.stderr).lower()

def detect_pydeface_mode():
    if PYDEFACE_RUNTIME == "local":
        return "local"
    if PYDEFACE_EXECUTABLE:
        return "exec_path"
    for mode in ("exec_name", "run"):
        if _probe(mode):
            return mode
    print("Could not launch pydeface from this image. Diagnostics:\n")
    sh([SING, "inspect", "--runscript", str(PYDEFACE_SIF)])
    sh([SING, "exec", "--cleanenv", str(PYDEFACE_SIF), "sh", "-c",
        "echo PATH=$PATH; find / -xdev -name 'pydeface*' 2>/dev/null | head -20"])
    raise RuntimeError("Set PYDEFACE_EXECUTABLE (path inside the image, from the output above) in the config cell, "
                       "or check that PYDEFACE_SIF really is a pydeface image.")

PYDEFACE_MODE = detect_pydeface_mode()
print("pydeface launch mode:", PYDEFACE_MODE, "\n")

def pydeface_cmd(infile, outfile):
    if PYDEFACE_MODE == "local":
        return ["pydeface", str(infile), "--outfile", str(outfile), "--force"]
    tail = [f"/work/{infile.name}", "--outfile", f"/work/{outfile.name}", "--force"]
    return _container_cmd(PYDEFACE_MODE, infile.parent, tail)

targets = []
for sub in SUBJECTS:
    for suf in DEFACE_SUFFIXES:
        targets += sorted((BIDS_OUT / f"sub-{sub}").rglob(f"anat/*_{suf}.nii.gz"))
print(f"{len(targets)} image(s) to consider\n")

if not LOG.exists():
    LOG.write_text("file\tdefaced_at\ttool\n")

for f in targets:
    rel = f.relative_to(BIDS_OUT)
    if str(rel) in done and not REDEFACE:
        print("already defaced, skipping:", rel); continue

    backup = BACKUP_DIR / rel
    backup.parent.mkdir(parents=True, exist_ok=True)
    if not backup.exists():                 # never overwrite the true original
        shutil.copy2(f, backup)

    tmp = f.with_name(f.name.replace(".nii.gz", "_defacetmp.nii.gz"))
    rc = sh(pydeface_cmd(f, tmp))
    if rc != 0 or not tmp.exists():
        print(f"\nFAILED: {rel}  (original left untouched)\n")
        tmp.unlink(missing_ok=True)
        continue
    os.replace(tmp, f)                      # defaced image takes the BIDS filename
    with LOG.open("a") as fh:
        fh.write(f"{rel}\t{datetime.datetime.now().isoformat(timespec='seconds')}\tpydeface ({PYDEFACE_RUNTIME})\n")
    print(f"\nDEFACED: {rel}\n")

In [ ]:
# Visual QC: original vs defaced (mid-sagittal + an axial slice through the face)
try:
    import nibabel as nib, numpy as np, matplotlib.pyplot as plt
except ImportError:
    print("Run:  %pip install --user nibabel matplotlib   (then restart the kernel and re-run this cell)")
else:
    pairs = [(BACKUP_DIR / f.relative_to(BIDS_OUT), f) for f in targets if (BACKUP_DIR / f.relative_to(BIDS_OUT)).exists()]
    if not pairs:
        print("Nothing to show yet.")
    else:
        fig, axes = plt.subplots(len(pairs), 4, figsize=(16, 4 * len(pairs)), squeeze=False)
        for row, (orig, new) in enumerate(pairs):
            for col, (path, label) in enumerate([(orig, "original"), (new, "defaced")]):
                data = nib.as_closest_canonical(nib.load(path)).get_fdata()
                vmax = np.percentile(data, 99.5)
                sag = data[data.shape[0] // 2, :, :].T
                axi = data[:, :, int(data.shape[2] * 0.35)].T
                for k, (sl, view) in enumerate([(sag, "sagittal"), (axi, "axial")]):
                    ax = axes[row, col * 2 + k]
                    ax.imshow(sl, cmap="gray", origin="lower", vmax=vmax)
                    ax.set_title(f"{label} - {view}\n{new.name[:42]}", fontsize=8)
                    ax.axis("off")
        plt.tight_layout(); plt.show()

## 8. Add the converted data to your main dataset (optional)

Copies only the `sub-*` folders from the staging folder into `MAIN_DATASET`, leaving `.heudiconv/` and anything else behind. It **never overwrites** existing files (`--ignore-existing`), so it won't touch the subject's `meg/` data. Leave `DRY_RUN = True` first to see what would be copied.

In [ ]:
DRY_RUN = True      # flip to False to actually copy

if MAIN_DATASET is None:
    print("MAIN_DATASET is None -> skipping. Set it in the config cell to use this step.")
else:
    MAIN_DATASET = Path(MAIN_DATASET)

    # Safety: refuse to copy identifiable anatomicals that haven't been defaced
    _log = BIDS_OUT / ".deface_log.tsv"
    _done = {l.split("\t")[0] for l in _log.read_text().splitlines()[1:]} if _log.exists() else set()
    _undefaced = [str(p.relative_to(BIDS_OUT)) for sub in SUBJECTS
                  for suf in ("T1w", "T2w", "FLAIR")
                  for p in (BIDS_OUT / f"sub-{sub}").rglob(f"anat/*_{suf}.nii.gz")
                  if str(p.relative_to(BIDS_OUT)) not in _done]
    if _undefaced and not DRY_RUN:
        raise RuntimeError("Not defaced yet (run Step 7 first):\n  " + "\n  ".join(_undefaced))
    elif _undefaced:
        print("WARNING - these have not been defaced and a real copy would be blocked:\n  " + "\n  ".join(_undefaced), "\n")

    for sub in SUBJECTS:
        src = BIDS_OUT / f"sub-{sub}"
        if not src.exists():
            print("nothing to copy for", src); continue
        dest = MAIN_DATASET / f"sub-{sub}"
        dest.mkdir(parents=True, exist_ok=True)
        cmd = ["rsync", "-av", "--ignore-existing"] + (["--dry-run"] if DRY_RUN else []) + [f"{src}/", f"{dest}/"]
        sh(cmd)
    if DRY_RUN:
        print("\nDry run only. Set DRY_RUN = False to copy.")

## 9. Validate with the BIDS validator

A standalone staging folder has no `dataset_description.json`, so the validator would report errors that don't matter. When `SKIP_TOP_LEVEL` is on, this step therefore validates **`MAIN_DATASET`** (after Step 8) rather than the staging folder.

In [ ]:
# Option A: local install (npm install -g bids-validator)
# Option B: Singularity -> first pull once:  singularity pull bids-validator.sif docker://bids/validator

TARGET = MAIN_DATASET if (SKIP_TOP_LEVEL and MAIN_DATASET) else BIDS_OUT
if SKIP_TOP_LEVEL and not MAIN_DATASET:
    print("Skipping: staging folder has no top-level files by design. Set MAIN_DATASET, copy (Step 8), then validate it.")
elif shutil.which("bids-validator"):
    sh(["bids-validator", TARGET])
elif SING and VALIDATOR_SIF.exists():
    sh([SING, "run", "--cleanenv", "-B", f"{Path(TARGET).resolve()}:/data:ro", VALIDATOR_SIF, "/data"])
else:
    print("No validator found. Options:\n"
          "  singularity pull bids-validator.sif docker://bids/validator   (then set VALIDATOR_SIF)\n"
          "  npm install -g bids-validator\n"
          "  or drop the folder into the browser validator (files stay local): https://bids-standard.github.io/bids-validator/")

## Troubleshooting

| Symptom | Likely fix |
|---|---|
| `0 files match` | Check `DICOM_TEMPLATE` and that the subject folder name matches `SUBJECTS` exactly. |
| `UNMATCHED series` | Add its `protocol_name` keywords to the heuristic rules. |
| `dataset_description.json` etc. appeared anyway | `SKIP_TOP_LEVEL` was `False`, or you re-ran into a folder that already had them. Delete them and re-run with it `True`. |\n| pydeface fails / `flirt: command not found` | pydeface needs FSL. Use the container (`PYDEFACE_RUNTIME = "singularity"`) or load FSL on the server before starting Jupyter. |\n| Defacing clipped brain tissue | Restore from `BACKUP_DIR`, set `REDEFACE = True`, and retry with other pydeface options (`--cost corratio`) or another tool (`mri_reface`). |\n| Output files named `run-01`, `run-02` for one scan | The series was split (e.g., repeated/incomplete run). Check `dicominfo.tsv` for duplicates; filter on `dim3`/`dim4`. |
| `Conversion directory already exists` / nothing re-converts | Set `OVERWRITE = True` (or delete `.heudiconv/<sub>`). |
| Multiple studies/sessions mixed in one folder | Add `--grouping accession_number` (or `study_uid`) to the command. |
| Subject ID rejected | BIDS labels are alphanumeric only. Use `--anon-cmd` or rename the folder. |
| Fieldmap missing `IntendedFor` | Check fieldmap series were matched and that `POPULATE_INTENDED_FOR_OPTS` is in your heuristic. |
| `FATAL: could not open image` / `command not found: singularity` | Check `SIF_PATH`; `module load singularity` (or `apptainer`) before launching Jupyter. |\n| Container can\'t see a folder (e.g. `/scratch`) | Add it to `EXTRA_BINDS`, and make sure `DICOM_ROOT` / `BIDS_OUT` use real paths (not symlinks that point outside bound folders). |\n| `Read-only file system` writing to `/home` | Singularity mounts `$HOME`; keep `BIDS_OUT` on project/scratch storage and bound via `EXTRA_BINDS` if needed. |

**Next steps:** add the new subjects to your main dataset's `participants.tsv` (HeuDiConv no longer touches it), and store `BACKUP_DIR` (the unfaced originals) with the same access controls as your raw DICOMs.